# Chapter 16 — Exercise Solutions## DPO and Preference Optimisation[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

### Solution 16.1: SimPO Implementation

In [ ]:
import torch
import torch.nn.functional as F

def simpo_loss(
    policy_chosen_logps: torch.Tensor,    # [batch] sum of log probs for chosen
    policy_rejected_logps: torch.Tensor,  # [batch] sum of log probs for rejected
    chosen_lengths: torch.Tensor,         # [batch] number of response tokens
    rejected_lengths: torch.Tensor,
    beta: float = 2.0,    # temperature for logit scaling
    gamma: float = 0.5,   # reward margin (ensures chosen > rejected by at least gamma)
) -> tuple:
    """
    SimPO: Simple Preference Optimisation (No reference model required).

    Key differences from DPO:
    1. Length-normalised log-probs (divide by sequence length)
    2. No reference model (no log-ratio against ref)
    3. Margin γ enforces minimum reward gap

    Loss: -E[log σ(β/|y_w| * log π(y_w|x) - β/|y_l| * log π(y_l|x) - γ)]
    """
    # Length normalisation: prevents length bias (long responses have lower absolute logprob)
    chosen_logps_norm   = policy_chosen_logps   / chosen_lengths
    rejected_logps_norm = policy_rejected_logps / rejected_lengths

    # Compute logit (before sigmoid)
    logits = beta * (chosen_logps_norm - rejected_logps_norm) - gamma

    # Loss: negative log-sigmoid
    loss = -F.logsigmoid(logits).mean()

    # Diagnostics
    accuracy = (chosen_logps_norm > rejected_logps_norm).float().mean()
    margin   = (chosen_logps_norm - rejected_logps_norm).mean()
    reward_chosen   = (beta * chosen_logps_norm).detach()
    reward_rejected = (beta * rejected_logps_norm).detach()

    return loss, accuracy, margin, reward_chosen, reward_rejected

# Compare DPO vs SimPO
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    chosen_ratio   = policy_chosen_logps   - ref_chosen_logps
    rejected_ratio = policy_rejected_logps - ref_rejected_logps
    logits = beta * (chosen_ratio - rejected_ratio)
    loss = -F.logsigmoid(logits).mean()
    accuracy = (chosen_ratio > rejected_ratio).float().mean()
    return loss, accuracy

# Synthetic batch
torch.manual_seed(42)
batch_size = 32
policy_chosen   = torch.randn(batch_size) * 0.5 - 4.0
policy_rejected = torch.randn(batch_size) * 0.5 - 5.0
ref_chosen      = torch.randn(batch_size) * 0.3 - 4.5
ref_rejected    = torch.randn(batch_size) * 0.3 - 4.8
chosen_lens     = torch.randint(50, 150, (batch_size,)).float()
rejected_lens   = torch.randint(50, 150, (batch_size,)).float()

simpo_l, simpo_acc, simpo_margin, _, _ = simpo_loss(
    policy_chosen, policy_rejected, chosen_lens, rejected_lens, beta=2.0, gamma=0.5)
dpo_l, dpo_acc = dpo_loss(
    policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta=0.1)

print(f"{'Method':<10} {'Loss':>8} {'Accuracy':>10}")
print("-" * 32)
print(f"{'DPO':<10} {dpo_l.item():>8.4f} {dpo_acc.item():>10.3f}")
print(f"{'SimPO':<10} {simpo_l.item():>8.4f} {simpo_acc.item():>10.3f}")
print()
print("SimPO advantages:")
print("  1. No reference model → ~2x less memory (no frozen model needed)")
print("  2. Length normalisation → unbiased toward response length")
print("  3. γ margin → enforces minimum confidence gap")
print("  4. Similar performance to DPO on most alignment benchmarks")
print("  5. Faster training (no ref forward pass each step)")

# YOUR TURN: What happens to SimPO loss when β=0? When γ=0?
# At what γ value does the loss become too restrictive (most pairs ignored)?


### Solution 16.2: Noise Robustness

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
import torch.nn.functional as F

def train_with_noise(noise_rate, n_steps=200, beta=0.1, batch_size=64):
    """Simulate DPO training with label noise."""
    torch.manual_seed(42); np.random.seed(42)
    # Start with random policy logprobs
    policy_param = torch.nn.Parameter(torch.randn(1) * 0.1)
    optimizer = torch.optim.Adam([policy_param], lr=1e-3)
    losses, accuracies = [], []

    for step in range(n_steps):
        # Generate batch: chosen should have higher quality (true signal)
        true_chosen   = policy_param + torch.randn(batch_size) * 0.3
        true_rejected = policy_param + torch.randn(batch_size) * 0.3 - 1.0
        ref_chosen    = torch.zeros(batch_size)
        ref_rejected  = torch.zeros(batch_size) - 1.0

        # Inject label noise: flip noise_rate fraction of labels
        noise_mask = torch.rand(batch_size) < noise_rate
        chosen   = torch.where(noise_mask, true_rejected, true_chosen)
        rejected = torch.where(noise_mask, true_chosen,   true_rejected)

        # DPO loss
        chosen_ratio   = chosen   - ref_chosen
        rejected_ratio = rejected - ref_rejected
        logits = beta * (chosen_ratio - rejected_ratio)
        loss = -F.logsigmoid(logits).mean()

        optimizer.zero_grad(); loss.backward(); optimizer.step()
        accuracy = (chosen_ratio > rejected_ratio).float().mean().item()
        losses.append(loss.item()); accuracies.append(accuracy)

    return np.array(losses), np.array(accuracies)

noise_rates = [0.0, 0.1, 0.2, 0.3]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['seagreen','steelblue','orange','tomato']

for noise, color in zip(noise_rates, colors):
    losses, accs = train_with_noise(noise)
    window = 20
    axes[0].plot(np.convolve(losses, np.ones(window)/window, 'valid'),
                 label=f'{noise*100:.0f}% noise', color=color, lw=2)
    axes[1].plot(np.convolve(accs,   np.ones(window)/window, 'valid'),
                 label=f'{noise*100:.0f}% noise', color=color, lw=2)

for ax, title in zip(axes, ['DPO Loss', 'Reward Accuracy']):
    ax.set_title(title); ax.set_xlabel('Training Step')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('DPO Robustness to Label Noise')
plt.tight_layout(); plt.show()

for noise in noise_rates:
    _, accs = train_with_noise(noise)
    print(f"Noise={noise*100:.0f}%: final accuracy={np.mean(accs[-20:]):.3f}")

print("\nKey finding: DPO degrades gracefully up to ~20% noise")
print("Beyond 30% noise: signal overwhelmed, accuracy near random (0.5)")
print("IPO (squared loss) is more robust to noise at cost of slower convergence")

# YOUR TURN: Implement IPO loss and compare at noise_rate=0.30
# IPO: L = E[(chosen_ratio - rejected_ratio - 1/(2β))²]
